<a href="https://colab.research.google.com/github/tabaraei/MS-Thesis/blob/main/Depression_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO General:
- Understand the division between train/test sets
- Obtain and save the labels for further usage

TODO Text (`XXX_TRANSCRIPT.csv`):
- Look for a proper model first
- See what the model requires as input (lemmatization, tokenization, vectorization)
- Loop over all the text files:
    - Prepare a standard text extracted from the CSV file
    - Normalize if needed
    - Feed the text to model
    - Append the extracted feature for each text to a feature set
- Save the feature embeddings for further usage

TODO Audio (`XXX_AUDIO.wav` at 16KHz):
- Look for a proper model first
- See what the model requires as input
- Loop over all the audio files:
    - Load the wav file
    - Perform normalization if needed
    - Feed the audio to model
    - Append the extracted feature for each audio to a feature set
- Save the feature embeddings for further usage

In [2]:
import os
import requests
import zipfile
from io import BytesIO
from google.colab import drive
from tqdm import tqdm

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/Data'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
class DAIC_WoZ:
    def __init__(self, DRIVE_PATH, download: bool=False):
        self.DRIVE_PATH = DRIVE_PATH
        self.DAIC_WoZ_DATA_PATH = f'{DRIVE_PATH}/DAIC-WoZ'
        self.DAIC_WoZ_DOWNLOAD_PATH = 'https://dcapswoz.ict.usc.edu/wwwdaicwoz/'
        self.file_names = ['AUDIO.wav', 'TRANSCRIPT.csv']
        self.select_sessions()
        if download: self.download_dataset()

    def select_sessions(self):
        all_sessions = numbers = set(range(300, 493))
        excluded_sessions = {342, 394, 398, 460}
        special_sessions = {373, 444, 451, 458, 480, 402}
        self.selected_sessions = all_sessions - excluded_sessions - special_sessions

    def download_dataset(self):
        os.makedirs(self.DAIC_WoZ_DATA_PATH, exist_ok=True)
        zip_files = [f'{session}_P.zip' for session in self.selected_sessions]
        for zip_name in tqdm(zip_files):
            response = requests.get(f'{self.DAIC_WoZ_DOWNLOAD_PATH}/{zip_name}', stream=True)
            response.raise_for_status()
            with zipfile.ZipFile(BytesIO(response.content)) as zf:
                prefix = zip_name[:3]
                for file_name in self.file_names:
                    zf.extract(f'{prefix}_{file_name}', path=self.DAIC_WoZ_DATA_PATH)


DAIC_dataset = DAIC_WoZ(DRIVE_PATH, download=True)

100%|██████████| 183/183 [1:45:42<00:00, 34.66s/it]
